# 05: Pruebas de UI Automatizadas con Playwright

Este notebook verifica que las interfaces de usuario de los servicios clave del Cognito-Stack estén activas y accesibles. Utiliza Playwright para lanzar un navegador, navegar a cada servicio y tomar una captura de pantalla.

**Servicios verificados:**
- **n8n:** Dashboard de workflows.
- **ComfyUI:** Interfaz de generación de imágenes.
- **Matrix Synapse:** Página de inicio (si es accesible).
- **Qdrant:** Dashboard de la base de datos de vectores.

In [ ]:
# Celda 1: Instalación y Configuración

from playwright.sync_api import sync_playwright, Page, expect
import time
import os

# Crear directorio para las capturas si no existe
SCREENSHOTS_DIR = '../screenshots'
os.makedirs(SCREENSHOTS_DIR, exist_ok=True)

print("Playwright importado y directorio de capturas listo.")

In [ ]:
# Celda 2: Función de Prueba Genérica

def test_service_ui(page: Page, name: str, url: str, selector: str, screenshot_filename: str):
    """Navega a la URL de un servicio, espera un selector y toma una captura."""
    print(f"--- Verificando UI de {name} en {url} ---")
    try:
        page.goto(url, wait_until='domcontentloaded', timeout=15000)
        # Esperar a que un elemento clave de la UI sea visible
        element = page.locator(selector)
        expect(element).to_be_visible(timeout=10000)
        time.sleep(2)  # Pausa para asegurar que la UI se renderice completamente
        
        screenshot_path = os.path.join(SCREENSHOTS_DIR, screenshot_filename)
        page.screenshot(path=screenshot_path)
        
        print(f"✓ {name} está accesible. Captura guardada en: {screenshot_path}")
        return True
    except Exception as e:
        print(f"❌ Falló la verificación de {name}: {str(e).splitlines()[0]}")
        return False

In [ ]:
# Celda 3: Ejecutar todas las pruebas de UI

def run_all_ui_tests():
    """Lanza el navegador y ejecuta las pruebas para cada servicio."""
    with sync_playwright() as p:
        # Usar --no-sandbox es a menudo necesario en entornos de contenedores
        browser = p.chromium.launch(headless=True, args=["--no-sandbox"])
        page = browser.new_page()
        
        print("🚀 Iniciando pruebas de UI...")
        
        # 1. Probar n8n
        test_service_ui(
            page=page,
            name="n8n",
            url="http://localhost:5678",
            selector="h1:has-text('Workflows')",
            screenshot_filename="01_n8n_dashboard.png"
        )
        
        # 2. Probar ComfyUI
        test_service_ui(
            page=page,
            name="ComfyUI",
            url="http://localhost:8188",
            selector="div#graph-canvas",
            screenshot_filename="02_comfyui_interface.png"
        )
        
        # 3. Probar Qdrant Dashboard
        test_service_ui(
            page=page,
            name="Qdrant Dashboard",
            url="http://localhost:6333/dashboard",
            selector="h2:has-text('Collections')",
            screenshot_filename="03_qdrant_dashboard.png"
        )
        
        # 4. Probar Matrix Synapse (página de inicio)
        test_service_ui(
            page=page,
            name="Matrix Synapse",
            url="http://localhost:8008",
            selector="body:has-text('It works! Synapse is running')",
            screenshot_filename="04_matrix_synapse_landing.png"
        )
        
        browser.close()
        print("\n✅ Pruebas de UI finalizadas.")

# Ejecutar la función principal
run_all_ui_tests()